Phase 2: 모델 구조 개선
Step 2.1: Deeper Classifier
- hidden*2 → hidden → num_classes
  를
- hidden*2 → hidden → hidden/2 → num_classes
  로 변경
검증: Validation F1 비교
Step 2.2: GCN Layer 추가
- GCN 2 layers → 3 layers
검증: Overfitting 여부 확인 (train vs val loss gap)
<!-- Step 2.3: Label Co-occurrence in Adjacency
- Adjacency matrix에 label co-occurrence 추가
검증: Macro F1 변화 관찰 -->

In [2]:
import os
import random
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Path Configuration
ROOT = r"F:/work/BIGdata/project_release/Amazon_products"
TRAIN_PATH = os.path.join(ROOT, "train/train_corpus.txt")
TEST_PATH = os.path.join(ROOT, "test/test_corpus.txt")
HIERARCHY_PATH = os.path.join(ROOT, "class_hierarchy.txt")
KEYWORDS_PATH = os.path.join(ROOT, "class_related_keywords.txt")
MODEL_SAVE_DIR = r'F:/work/BIGdata/project_release/model'

# LLM generated data paths
LLM_DATA_PATH = os.path.join(ROOT, "llm_generated_data.pkl")
FINAL_LABELS_PATH = os.path.join(MODEL_SAVE_DIR, "final_train_labels.pkl")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [3]:
# --- Data Loading ---
def load_data_and_graph():
    documents = []
    with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t', 1)
            if len(parts) == 2:
                documents.append({'id': int(parts[0]), 'text': parts[1]})
    
    test_documents = []
    if os.path.exists(TEST_PATH):
        with open(TEST_PATH, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t', 1)
                if len(parts) == 2:
                    test_documents.append({'id': int(parts[0]), 'text': parts[1]})
    
    class_info = {}
    with open(KEYWORDS_PATH, 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            parts = line.strip().split(':')
            if len(parts) == 2:
                name = parts[0].strip().replace('_', ' ')
                keywords = parts[1].strip()
                class_info[idx] = {'name': name, 'keywords': keywords}
    
    G = nx.DiGraph() #Directional Graph
    with open(HIERARCHY_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            p, c = map(int, line.strip().split())
            G.add_edge(p, c) #p->c 방향으로 edge 형성 (부모->자식)
    
    num_classes = len(class_info)
    roots = [n for n, d in G.in_degree() if d == 0]
    
    return documents, test_documents, G, class_info, num_classes, roots

all_docs, test_docs, G, class_info, num_classes, roots = load_data_and_graph()
print(f"Loaded {len(all_docs)} training docs, {len(test_docs)} test docs, and {num_classes} classes")

Loaded 29487 training docs, 19658 test docs, and 531 classes


In [4]:
# --- Load Pre-generated LLM Labels ---

def load_llm_labels():
    val_data = None
    seed_train_labels = None
    unlabeled_indices = None
    final_train_labels = None
    
    if os.path.exists(LLM_DATA_PATH):
        print(f"Loading LLM labels from {LLM_DATA_PATH}...")
        with open(LLM_DATA_PATH, 'rb') as f:
            data = pickle.load(f)
            val_data = data.get('val_data', [])
            seed_train_labels = data.get('seed_train_labels', {})
            unlabeled_indices = data.get('unlabeled_indices', [])
        
        print(f"  - Validation data: {len(val_data)} samples")
        print(f"  - Seed train labels: {len(seed_train_labels)} samples")
        print(f"  - Unlabeled indices: {len(unlabeled_indices)} samples")
    
    if os.path.exists(FINAL_LABELS_PATH):
        print(f"\nLoading final train labels from {FINAL_LABELS_PATH}...")
        with open(FINAL_LABELS_PATH, 'rb') as f:
            final_train_labels = pickle.load(f)
        print(f"  - Final train labels: {len(final_train_labels)} samples")
    
    return val_data, seed_train_labels, unlabeled_indices, final_train_labels

val_data, seed_train_labels, unlabeled_indices, final_train_labels = load_llm_labels()

if final_train_labels is not None:
    print("\n==> Using final_train_labels for training")
    use_final_labels = True
elif seed_train_labels is not None:
    print("\n==> Using seed_train_labels for training")
    use_final_labels = False
else:
    raise ValueError("No pre-generated labels found!")

Loading LLM labels from F:/work/BIGdata/project_release/Amazon_products\llm_generated_data.pkl...
  - Validation data: 200 samples
  - Seed train labels: 600 samples
  - Unlabeled indices: 28687 samples

==> Using seed_train_labels for training


In [5]:
# --- Hierarchical Constraint Functions ---
# 자식 label을 가지면 반드시 조상 label도 갖게하는 역할 : 트리 구조의 labeling에서 부모 클래스의 누락을 방지

def get_ancestors(G, node):
    ancestors = set()
    queue = [node]
    while queue:
        current = queue.pop(0)
        for parent in G.predecessors(current):
            if parent not in ancestors:
                ancestors.add(parent)
                queue.append(parent)
    return ancestors

def enforce_hierarchy_constraint(labels, G):
    constrained_labels = set(labels)
    for label in labels:
        ancestors = get_ancestors(G, label)
        constrained_labels.update(ancestors)
    return list(constrained_labels)

# Apply hierarchy constraint
print("\nApplying hierarchy constraints...")

if use_final_labels:
    for doc_id in final_train_labels:
        original = final_train_labels[doc_id]
        constrained = enforce_hierarchy_constraint(original, G)
        final_train_labels[doc_id] = constrained
    print(f"Applied to {len(final_train_labels)} final labels")
else:
    for idx in seed_train_labels:
        original = seed_train_labels[idx]
        constrained = enforce_hierarchy_constraint(original, G)
        seed_train_labels[idx] = constrained
    print(f"Applied to {len(seed_train_labels)} seed labels")

if val_data:
    for item in val_data:
        original = item['labels']
        constrained = enforce_hierarchy_constraint(original, G)
        item['labels'] = constrained
    print(f"Applied to {len(val_data)} validation labels")


Applying hierarchy constraints...
Applied to 600 seed labels
Applied to 200 validation labels


In [6]:
# --- Silver Labeling ---
"""
클래스 설명(class description) 생성
라벨이 없는 문서의 텍스트 추출
TF-IDF 학습 후 문서/클래스 벡터 변환
cosine similarity로 문서-클래스 매칭
상위 Top-K 클래스를 선택
계층 라벨 보정(hierarchy constraint 적용)
최종 라벨을 silver_labels에 저장
"""
def generate_silver_labels_tfidf(documents, unlabeled_indices, class_info, num_classes):
    print("\nGenerating silver labels...")
    
    class_descriptions = []
    for i in range(num_classes):
        desc = f"{class_info[i]['name']} {class_info[i]['keywords']}"
        class_descriptions.append(desc)
    
    unlabeled_docs = [documents[i] for i in unlabeled_indices]
    doc_texts = [d['text'] for d in unlabeled_docs]
    
    vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
    all_texts = doc_texts + class_descriptions
    vectorizer.fit(all_texts)
    
    doc_vectors = vectorizer.transform(doc_texts)
    class_vectors = vectorizer.transform(class_descriptions)
    similarity = cosine_similarity(doc_vectors, class_vectors)
    
    silver_labels = {}
    for doc_idx, unlabeled_idx in enumerate(unlabeled_indices):
        top_k = 5
        top_classes = np.argsort(similarity[doc_idx])[-top_k:].tolist()
        threshold = np.percentile(similarity[doc_idx], 85)
        top_classes = [c for c in top_classes if similarity[doc_idx][c] > threshold]
        
        if len(top_classes) < 2:
            top_classes = np.argsort(similarity[doc_idx])[-2:].tolist()
        elif len(top_classes) > 3:
            top_classes = top_classes[-3:]
        
        top_classes = enforce_hierarchy_constraint(top_classes, G)
        
        if len(top_classes) > 3:
            scores = [(c, similarity[doc_idx][c]) for c in top_classes]
            scores.sort(key=lambda x: x[1], reverse=True)
            top_classes = [c for c, _ in scores[:3]]
        
        silver_labels[unlabeled_idx] = top_classes
    
    return silver_labels

if not use_final_labels and unlabeled_indices:
    silver_labels = generate_silver_labels_tfidf(all_docs, unlabeled_indices, class_info, num_classes)
    print(f"Generated silver labels for {len(silver_labels)} documents")
    combined_train_labels = {**seed_train_labels, **silver_labels}
    print(f"Total training labels: {len(combined_train_labels)}")
else:
    combined_train_labels = final_train_labels
    print(f"Using {len(combined_train_labels)} final training labels")


Generating silver labels...
Generated silver labels for 28687 documents
Total training labels: 29287


In [7]:
# --- Build Adjacency Matrix ---

def build_adjacency_matrix(G, num_classes):
    A = torch.eye(num_classes)
    for u, v in G.edges():
        A[u, v] = 1
        A[v, u] = 1
    
    degree = A.sum(dim=1)
    degree_inv_sqrt = torch.pow(degree, -0.5)
    degree_inv_sqrt[torch.isinf(degree_inv_sqrt)] = 0
    D_inv_sqrt = torch.diag(degree_inv_sqrt)
    A_hat = D_inv_sqrt @ A @ D_inv_sqrt
    
    return A_hat

A_hat = build_adjacency_matrix(G, num_classes).to(device)
print(f"Adjacency matrix shape: {A_hat.shape}")

Adjacency matrix shape: torch.Size([531, 531])


In [8]:
# --- Dataset ---

class TextDataset(Dataset):
    def __init__(self, texts, labels_dict, tokenizer, num_classes, max_length=128):
        self.texts = texts
        self.labels_dict = labels_dict
        self.tokenizer = tokenizer
        self.num_classes = num_classes
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        label_vector = torch.zeros(self.num_classes)
        if idx in self.labels_dict:
            for label in self.labels_dict[idx]:
                if label < self.num_classes:
                    label_vector[label] = 1
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': label_vector
        }

In [9]:
# --- Model ---
# --- IMPROVED Model with Hierarchical Attention ---

class HierarchicalGCNClassifier(nn.Module):
    """
    IMPROVED model with:
    1. Multiple GCN layers with residual connections @
    2. Hierarchical attention mechanism
    3. Label-wise attention
    4. Dropout regularization
    """
    def __init__(self, num_classes, bert_model='bert-base-uncased', hidden_dim=768, gcn_layers=3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_model)
        self.num_classes = num_classes
        self.hidden_dim = hidden_dim
        
        # IMPROVED: More GCN layers with residual connections
        self.gcn_layers = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim) for _ in range(gcn_layers)
        ])
        self.gcn_norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim) for _ in range(gcn_layers)
        ])
        
        # Learnable class embeddings
        self.class_embeddings = nn.Parameter(torch.randn(num_classes, hidden_dim) * 0.02)
        
        # Multi-head attention for document-class interaction
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=8, batch_first=True, dropout=0.1)
        
        # IMPROVED: Label-wise attention
        self.label_attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        # IMPROVED: Deeper classifier with residual
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, num_classes)
        )
        
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, input_ids, attention_mask, A_hat):
        # BERT encoding
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Use mean pooling instead of just [CLS] token
        hidden_states = bert_output.last_hidden_state
        mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        sum_hidden = torch.sum(hidden_states * mask_expanded, dim=1)
        sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
        doc_embed = sum_hidden / sum_mask
        
        # GCN for class embeddings with residual connections
        class_feats = self.class_embeddings
        for gcn_layer, gcn_norm in zip(self.gcn_layers, self.gcn_norms):
            residual = class_feats
            class_feats = gcn_layer(A_hat @ class_feats)
            class_feats = F.gelu(class_feats)
            class_feats = gcn_norm(class_feats)
            class_feats = class_feats + residual  # Residual connection
        
        class_feats = self.dropout(class_feats)
        
        # Document-class attention
        doc_expanded = doc_embed.unsqueeze(1)
        class_expanded = class_feats.unsqueeze(0).expand(doc_embed.size(0), -1, -1)
        
        attn_output, attn_weights = self.attention(doc_expanded, class_expanded, class_expanded)
        attn_output = attn_output.squeeze(1)
        
        # Combine document and attention features
        combined = torch.cat([doc_embed, attn_output], dim=1)
        logits = self.classifier(combined)
        
        return logits

# class ImprovedGCNClassifier(nn.Module):
#     def __init__(self, num_classes, bert_model='bert-base-uncased', hidden_dim=768, gcn_layers=2):
#         super().__init__()
#         self.bert = AutoModel.from_pretrained(bert_model)
#         self.num_classes = num_classes
#         self.hidden_dim = hidden_dim
        
#         self.gcn_layers = nn.ModuleList([
#             nn.Linear(hidden_dim, hidden_dim) for _ in range(gcn_layers)
#         ])
        
#         self.class_embeddings = nn.Parameter(torch.randn(num_classes, hidden_dim))
#         self.attention = nn.MultiheadAttention(hidden_dim, num_heads=8, batch_first=True)
        
#         self.classifier = nn.Sequential(
#             nn.Linear(hidden_dim * 2, hidden_dim),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(hidden_dim, num_classes)
#         )
        
#         self.layer_norm = nn.LayerNorm(hidden_dim)
    
#     def forward(self, input_ids, attention_mask, A_hat):
#         bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
#         doc_embed = bert_output.last_hidden_state[:, 0, :]
        
#         class_feats = self.class_embeddings
#         for gcn_layer in self.gcn_layers:
#             class_feats = F.relu(gcn_layer(A_hat @ class_feats))
#             class_feats = self.layer_norm(class_feats)
        
#         doc_expanded = doc_embed.unsqueeze(1)
#         class_expanded = class_feats.unsqueeze(0).expand(doc_embed.size(0), -1, -1)
        
#         attn_output, _ = self.attention(doc_expanded, class_expanded, class_expanded)
#         attn_output = attn_output.squeeze(1)
        
#         combined = torch.cat([doc_embed, attn_output], dim=1)
#         logits = self.classifier(combined)
        
#         return logits

In [10]:
# --- Loss and Training ---

#=========================================================================# step1. improved loss: combin Asymmetric loss, Focalloss
class AsymmetricLoss(nn.Module):
    """
    IMPROVED: Asymmetric Loss for multi-label classification
    Better handles class imbalance than Focal Loss
    """
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
    
    def forward(self, logits, targets):
        # Sigmoid probabilities
        probs = torch.sigmoid(logits)
        
        # Asymmetric clipping
        probs_neg = probs.clamp(min=self.clip)
        
        # Positive and negative losses
        pos_loss = targets * torch.log(probs.clamp(min=1e-8))
        neg_loss = (1 - targets) * torch.log((1 - probs_neg).clamp(min=1e-8))
        
        # Asymmetric focusing
        pos_weight = (1 - probs) ** self.gamma_pos
        neg_weight = probs ** self.gamma_neg
        
        loss = -pos_weight * pos_loss - neg_weight * neg_loss
        return loss.mean()

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, logits, targets):
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

class CombinedLoss(nn.Module):
    """IMPROVED: Combined loss for better training"""
    def __init__(self, alpha=0.8):
        super().__init__()
        self.focal = FocalLoss(alpha=0.25, gamma=2.0)
        self.asymmetric = AsymmetricLoss(gamma_neg=3, gamma_pos=1,clip=0.03)
        self.alpha = alpha
    
    def forward(self, logits, targets):
        return self.alpha * self.focal(logits, targets) + (1 - self.alpha) * self.asymmetric(logits, targets)
#====================================================#
def train_epoch(model, loader, optimizer, criterion, A_hat, device):
    model.train()
    total_loss = 0
    
    for batch in tqdm(loader, desc="Training"):
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        
        optimizer.zero_grad()
        logits = model(ids, mask, A_hat)
        loss = criterion(logits, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def evaluate(model, loader, A_hat, device, threshold=0.5):
    model.eval()
    preds, targets = [], []
    
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['labels'].to(device)
            
            logits = model(ids, mask, A_hat)
            pred = (torch.sigmoid(logits) > threshold).float()
            
            preds.append(pred.cpu().numpy())
            targets.append(lbls.cpu().numpy())

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    
    tp = np.sum(preds * targets)
    fp = np.sum(preds * (1 - targets))
    fn = np.sum((1 - preds) * targets)
    micro_f1 = 2 * tp / (2 * tp + fp + fn + 1e-10)
    
    tp_per_class = np.sum(preds * targets, axis=0)
    fp_per_class = np.sum(preds * (1 - targets), axis=0)
    fn_per_class = np.sum((1 - preds) * targets, axis=0)
    f1_per_class = 2 * tp_per_class / (2 * tp_per_class + fp_per_class + fn_per_class + 1e-10)
    macro_f1 = np.mean(f1_per_class)
    
    return micro_f1, macro_f1

class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.001, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.best_loss = None
        self.early_stop = False
        self.count = 0

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
            return

        # loss improvement 없음
        if self.best_loss - val_loss < self.min_delta:
            self.count += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.count}/{self.patience}")
            if self.count >= self.patience:
                self.early_stop = True
        else:
            # loss 개선됨
            self.best_loss = val_loss
            self.count = 0


In [11]:
# --- Prepare Data ---

id_to_doc = {d['id']: d for d in all_docs}

train_texts = []
train_labels_remapped = {}

for key, labels in combined_train_labels.items():
    if isinstance(key, int):
        if key in id_to_doc:
            doc = id_to_doc[key]
            train_texts.append(doc['text'])
            train_labels_remapped[len(train_texts) - 1] = labels
        elif 0 <= key < len(all_docs):
            doc = all_docs[key]
            train_texts.append(doc['text'])
            train_labels_remapped[len(train_texts) - 1] = labels

print(f"Prepared {len(train_texts)} training samples")

if val_data:
    val_texts = [item['text'] for item in val_data]
    val_labels_remapped = {i: item['labels'] for i, item in enumerate(val_data)}
    print(f"Prepared {len(val_texts)} validation samples")
else:
    from sklearn.model_selection import train_test_split
    train_indices = list(range(len(train_texts)))
    train_idx, val_idx = train_test_split(train_indices, test_size=0.2, random_state=42)
    
    val_texts = [train_texts[i] for i in val_idx]
    val_labels_remapped = {i: train_labels_remapped[val_idx[i]] for i in range(len(val_idx))}
    
    train_texts = [train_texts[i] for i in train_idx]
    new_train_labels = {i: train_labels_remapped[train_idx[i]] for i in range(len(train_idx))}
    train_labels_remapped = new_train_labels
    
    print(f"Split: {len(train_texts)} train, {len(val_texts)} val")

Prepared 28687 training samples
Prepared 200 validation samples


In [12]:
# -- training --
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

train_ds = TextDataset(train_texts, train_labels_remapped, tokenizer, num_classes)
val_ds = TextDataset(val_texts, val_labels_remapped, tokenizer, num_classes)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

model = HierarchicalGCNClassifier(num_classes, gcn_layers=3).to(device)

bert_params = list(model.bert.parameters())
other_params = [p for n, p in model.named_parameters() if 'bert' not in n]

optimizer = torch.optim.AdamW([
    {'params': bert_params, 'lr': 1e-5},
    {'params': other_params, 'lr': 2e-4}
], weight_decay=0.01)
#optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01) --> bert, classfier 다른 lr # ===== #


# warmup+Cosine Annealing scheduler #=====#
# Use combined loss #=====#
criterion = CombinedLoss(alpha=0.7)

# IMPROVED: Warmup + Cosine Annealing #=====#
total_steps = len(train_loader) * 15  # Total training steps
warmup_steps = len(train_loader) * 2  # 2 epochs warmup

best_f1 = 0
SELF_TRAIN_ROUNDS = 3
EPOCHS_PER_ROUND = 5

# def lr_lambda(current_step):
#     if current_step < warmup_steps:
#         return float(current_step) / float(max(1, warmup_steps))
#     progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
#     return max(0.1, 0.5 * (1.0 + np.cos(np.pi * progress)))
#scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
criterion = CombinedLoss(alpha=0.7) 
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SELF_TRAIN_ROUNDS * EPOCHS_PER_ROUND)



print("\n=== Starting Training ===")

for round_num in range(SELF_TRAIN_ROUNDS):
    print(f"\n{'='*50}")
    print(f"Round {round_num + 1}/{SELF_TRAIN_ROUNDS}")
    print(f"{'='*50}")
    
    # Lower starting threshold
    confidence_threshold = 0.55 - (round_num * 0.05)  # 0.55 -> 0.50 -> 0.45
    print(f"Confidence threshold: {confidence_threshold:.2f}")
    
    for epoch in range(EPOCHS_PER_ROUND):
        loss = train_epoch(model, train_loader, optimizer, criterion, A_hat, device)
        micro_f1, macro_f1 = evaluate(model, val_loader, A_hat, device)
        scheduler.step()
        
        print(f"  Epoch {epoch+1}/{EPOCHS_PER_ROUND}: Loss={loss:.4f}, "
              f"Micro-F1={micro_f1:.4f}, Macro-F1={macro_f1:.4f}")
        
        if micro_f1 > best_f1:
            best_f1 = micro_f1
            model_path = os.path.join(MODEL_SAVE_DIR, "best_model_step1.pt") #################################fix here#####################
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'f1': best_f1,
                'round': round_num,
                'epoch': epoch
            }, model_path)
            print(f"    -> Best model saved! F1={best_f1:.4f}")

    if round_num < SELF_TRAIN_ROUNDS - 1:
        print("\n  Updating pseudo-labels...")
        model.eval()
        
        inference_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
        new_labels = {}
        empty_fix_count = 0
        confident_update_count = 0
        
        with torch.no_grad():
            batch_idx = 0
            for batch in tqdm(inference_loader, desc="Inference"):
                ids = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                logits = model(ids, mask, A_hat)
                probs = torch.sigmoid(logits).cpu()
                
                for i, prob_vec in enumerate(probs):
                    global_idx = batch_idx + i
                    
                    # Get original labels (CRITICAL)
                    original_labels = train_labels_remapped.get(global_idx, [])
                    
                    # Get high confidence predictions
                    high_conf = (prob_vec > confidence_threshold).nonzero(as_tuple=True)[0].tolist()
                    
                    # Strategy: Only update if confident AND meets label count requirement
                    if len(high_conf) >= 2:  # At least 2 confident predictions
                        # Apply hierarchy constraint
                        high_conf = enforce_hierarchy_constraint(high_conf, G)
                        
                        # Limit to top 3
                        if len(high_conf) > 3:
                            top_probs = [(c, prob_vec[c].item()) for c in high_conf]
                            top_probs.sort(key=lambda x: x[1], reverse=True)
                            high_conf = [c for c, _ in top_probs[:3]]
                        
                        new_labels[global_idx] = high_conf
                        confident_update_count += 1
                    
                    else:
                        if len(original_labels) >= 2:
                            new_labels[global_idx] = original_labels
                        else:
                            # Last resort: use top-2 predictions even if not confident
                            top_indices = np.argsort(prob_vec.numpy())[-3:].tolist()
                            top_indices = enforce_hierarchy_constraint(top_indices, G)
                            
                            # Ensure 2-3 labels
                            if len(top_indices) > 3:
                                top_probs = [(c, prob_vec[c].item()) for c in top_indices]
                                top_probs.sort(key=lambda x: x[1], reverse=True)
                                top_indices = [c for c, _ in top_probs[:3]]
                            elif len(top_indices) < 2:
                                # Absolute fallback
                                top_indices = np.argsort(prob_vec.numpy())[-2:].tolist()
                            
                            new_labels[global_idx] = top_indices
                            empty_fix_count += 1
                
                batch_idx += len(prob_vec)
        
        # Final safety check: ensure ALL indices have labels
        for idx in range(len(train_texts)):
            if idx not in new_labels or len(new_labels[idx]) == 0:
                if idx in train_labels_remapped and len(train_labels_remapped[idx]) > 0:
                    new_labels[idx] = train_labels_remapped[idx]
                else:
                    # Emergency: assign most common labels
                    new_labels[idx] = [0, 1]  # Root classes as fallback
                empty_fix_count += 1
        
        # Verify label quality
        empty_count = sum(1 for v in new_labels.values() if len(v) == 0)
        avg_labels = np.mean([len(v) for v in new_labels.values()])
        changed = sum(1 for k in new_labels if new_labels[k] != train_labels_remapped.get(k, []))
        
        print(f"  Confident updates: {confident_update_count}")
        print(f"  Empty label fixes: {empty_fix_count}")
        print(f"  Total changed: {changed}/{len(new_labels)}")
        print(f"  Avg labels per doc: {avg_labels:.2f}")
        
        if empty_count > 0:
            print(f"WARNING: {empty_count} samples still have empty labels!")
        else:
            print(f"All samples have valid labels")
        
        # Update dataset
        train_labels_remapped = new_labels
        train_ds = TextDataset(train_texts, train_labels_remapped, tokenizer, num_classes)
        train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

print(f"\n=== Training Complete. Best Micro F1: {best_f1:.4f} ===")


=== Starting Training ===

Round 1/3
Confidence threshold: 0.55


Training: 100%|████████████████████████████████████████████████████████████████████| 1793/1793 [07:01<00:00,  4.26it/s]


  Epoch 1/5: Loss=0.0037, Micro-F1=0.1167, Macro-F1=0.0464
    -> Best model saved! F1=0.1167


Training: 100%|████████████████████████████████████████████████████████████████████| 1793/1793 [06:59<00:00,  4.27it/s]


  Epoch 2/5: Loss=0.0018, Micro-F1=0.1268, Macro-F1=0.0543
    -> Best model saved! F1=0.1268


...

  Epoch 3/5: Loss=0.0011, Micro-F1=0.1284, Macro-F1=0.0616


Training: 100%|████████████████████████████████████████████████████████████████████| 1793/1793 [07:07<00:00,  4.20it/s]


  Epoch 4/5: Loss=0.0010, Micro-F1=0.1291, Macro-F1=0.0594


Training:  34%|███████████████████████▌                                             | 612/1793 [02:29<04:49,  4.08it/s]


KeyboardInterrupt: 

In [14]:
"""
Generate Kaggle Submission using trained model (best_model_step1.pt)
"""

import os
import csv
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import networkx as nx
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

# ------------------------
# Paths
# ------------------------
ROOT = r"F:/work/BIGdata/project_release/Amazon_products"
TEST_PATH = os.path.join(ROOT, "test/test_corpus.txt")
HIERARCHY_PATH = os.path.join(ROOT, "class_hierarchy.txt")
KEYWORDS_PATH = os.path.join(ROOT, "class_related_keywords.txt")
MODEL_SAVE_DIR = r'F:/work/BIGdata/project_release/model'
MODEL_PATH = os.path.join(MODEL_SAVE_DIR, "best_model_step11.pt") ###########################fix here#####################################

SUBMISSION_DIR = r"./BIGdata/project_release/submission"
SUBMISSION_PATH = os.path.join(SUBMISSION_DIR, "submission.csv")
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# # ------------------------
# # Build Adjacency Matrix
# # ------------------------
# def build_adjacency_matrix(G, num_classes):
#     A = torch.eye(num_classes)
#     for u, v in G.edges():
#         A[u, v] = 1
#         A[v, u] = 1
    
#     degree = A.sum(dim=1)
#     degree_inv_sqrt = torch.pow(degree, -0.5)
#     degree_inv_sqrt[torch.isinf(degree_inv_sqrt)] = 0
#     D_inv_sqrt = torch.diag(degree_inv_sqrt)
#     A_hat = D_inv_sqrt @ A @ D_inv_sqrt
    
#     return A_hat

# A_hat = build_adjacency_matrix(G, num_classes).to(device)

# ------------------------
# Model Definition
# ------------------------
# class ImprovedGCNClassifier(nn.Module):
#     def __init__(self, num_classes, bert_model='bert-base-uncased', hidden_dim=768, gcn_layers=2):
#         super().__init__()
#         self.bert = AutoModel.from_pretrained(bert_model)
#         self.num_classes = num_classes
#         self.hidden_dim = hidden_dim
        
#         self.gcn_layers = nn.ModuleList([
#             nn.Linear(hidden_dim, hidden_dim) for _ in range(gcn_layers)
#         ])
        
#         self.class_embeddings = nn.Parameter(torch.randn(num_classes, hidden_dim))
#         self.attention = nn.MultiheadAttention(hidden_dim, num_heads=8, batch_first=True)
        
#         self.classifier = nn.Sequential(
#             nn.Linear(hidden_dim * 2, hidden_dim),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(hidden_dim, num_classes)
#         )
        
#         self.layer_norm = nn.LayerNorm(hidden_dim)
    
#     def forward(self, input_ids, attention_mask, A_hat):
#         bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
#         doc_embed = bert_output.last_hidden_state[:, 0, :]
        
#         class_feats = self.class_embeddings
#         for gcn_layer in self.gcn_layers:
#             class_feats = F.relu(gcn_layer(A_hat @ class_feats))
#             class_feats = self.layer_norm(class_feats)
        
#         doc_expanded = doc_embed.unsqueeze(1)
#         class_expanded = class_feats.unsqueeze(0).expand(doc_embed.size(0), -1, -1)
        
#         attn_output, _ = self.attention(doc_expanded, class_expanded, class_expanded)
#         attn_output = attn_output.squeeze(1)
        
#         combined = torch.cat([doc_embed, attn_output], dim=1)
#         logits = self.classifier(combined)
        
#         return logits

#------------------------
#Dataset
#------------------------
class TestDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0)
        }

# ------------------------
# Load Model
# ------------------------
print(f"\nLoading model from {MODEL_PATH}...")
model = HierarchicalGCNClassifier(num_classes, gcn_layers=3).to(device)
checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Model loaded successfully! (Best F1: {checkpoint.get('f1', 'N/A')})")

# ------------------------
# Generate Predictions
# ------------------------
print("\nGenerating predictions...")
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

test_texts = [doc['text'] for doc in test_docs]
test_dataset = TestDataset(test_texts, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

all_predictions = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        logits = model(ids, mask, A_hat)
        probs = torch.sigmoid(logits).cpu().numpy()
        
        for prob_vec in probs:
            # Get top predictions
            top_indices = np.argsort(prob_vec)[-5:][::-1]  # Top 5 candidates
            
            # Apply threshold
            threshold = 0.3
            predictions = [idx for idx in top_indices if prob_vec[idx] > threshold]
            
            # Ensure 1-3 labels (task requirement)
            if len(predictions) < 1:
                predictions = [top_indices[0]]  # At least 1
            elif len(predictions) > 3:
                predictions = predictions[:3]  # At most 3
            
            # Apply hierarchy constraint
            predictions = enforce_hierarchy_constraint(predictions, G)
            
            # Final limit to 3 most confident
            if len(predictions) > 3:
                top_probs = [(p, prob_vec[p]) for p in predictions]
                top_probs.sort(key=lambda x: x[1], reverse=True)
                predictions = [p for p, _ in top_probs[:3]]
            
            # Ensure at least 1 label
            if len(predictions) == 0:
                predictions = [top_indices[0]]
            
            all_predictions.append(sorted(predictions))

# ------------------------
# Save Submission
# ------------------------
print(f"\nSaving submission to {SUBMISSION_PATH}...")

with open(SUBMISSION_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'labels'])
    
    for doc, preds in zip(test_docs, all_predictions):
        pid = doc['id']
        labels_str = ','.join(map(str, preds))
        writer.writerow([pid, labels_str])

print("[SUCCESS] Submission file saved!")
print(f"  Path: {SUBMISSION_PATH}")
print(f"  Total samples: {len(all_predictions)}")
print(f"  Avg labels per sample: {np.mean([len(p) for p in all_predictions]):.2f}")
print(f"  Label distribution:")
print(f"    - 1 label: {sum(1 for p in all_predictions if len(p) == 1)}")
print(f"    - 2 labels: {sum(1 for p in all_predictions if len(p) == 2)}")
print(f"    - 3 labels: {sum(1 for p in all_predictions if len(p) == 3)}")
print(f"    - 3+ labels: {sum(1 for p in all_predictions if len(p) > 3)}")

print("\n[SUCCESS] Done! Ready for Kaggle submission.")

Using device: cuda

Loading model from F:/work/BIGdata/project_release/model\best_model_step11.pt...


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_11752\1075917185.py:131: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(MODEL_PATH, map_locati

Model loaded successfully! (Best F1: 0.13997627520758363)

Generating predictions...


Predicting: 100%|████████████████████████████████████████████████████████████████████| 615/615 [01:26<00:00,  7.15it/s]


Saving submission to ./BIGdata/project_release/submission\submission.csv...
[SUCCESS] Submission file saved!
  Path: ./BIGdata/project_release/submission\submission.csv
  Total samples: 19658
  Avg labels per sample: 3.00
  Label distribution:
    - 1 label: 0
    - 2 labels: 3
    - 3 labels: 19655
    - 3+ labels: 0

[SUCCESS] Done! Ready for Kaggle submission.


#### 

In [48]:
# 모든 leaf node 찾기: out-degree = 0
leaf_nodes = [n for n in G.nodes() if G.out_degree(n) == 0]

# leaf별 조상 수 계산
leaf_ancestor_count = {}

for leaf in leaf_nodes:
    ancestors = nx.ancestors(G, leaf)  # leaf의 모든 조상 set
    leaf_ancestor_count[leaf] = len(ancestors)

print("=== Leaf node ancestor counts ===")
for leaf, cnt in leaf_ancestor_count.items():
    print(f"Leaf {leaf}: {cnt} ancestors")

# 전체 leaf에서의 평균 조상 수
avg_ancestors = sum(leaf_ancestor_count.values()) / len(leaf_ancestor_count)
print(f"\nAverage number of ancestors for leaf nodes: {avg_ancestors:.2f}")


=== Leaf node ancestor counts ===
Leaf 2: 2 ancestors
Leaf 434: 2 ancestors
Leaf 413: 3 ancestors
Leaf 483: 2 ancestors
Leaf 464: 3 ancestors
Leaf 520: 2 ancestors
Leaf 527: 2 ancestors
Leaf 7: 2 ancestors
Leaf 19: 2 ancestors
Leaf 70: 2 ancestors
Leaf 174: 2 ancestors
Leaf 176: 2 ancestors
Leaf 183: 2 ancestors
Leaf 188: 2 ancestors
Leaf 210: 2 ancestors
Leaf 238: 2 ancestors
Leaf 294: 2 ancestors
Leaf 375: 2 ancestors
Leaf 392: 2 ancestors
Leaf 407: 2 ancestors
Leaf 433: 2 ancestors
Leaf 435: 2 ancestors
Leaf 6: 2 ancestors
Leaf 57: 2 ancestors
Leaf 97: 2 ancestors
Leaf 133: 2 ancestors
Leaf 228: 2 ancestors
Leaf 331: 2 ancestors
Leaf 385: 2 ancestors
Leaf 9: 2 ancestors
Leaf 206: 2 ancestors
Leaf 235: 2 ancestors
Leaf 270: 2 ancestors
Leaf 285: 2 ancestors
Leaf 308: 2 ancestors
Leaf 327: 2 ancestors
Leaf 391: 2 ancestors
Leaf 405: 2 ancestors
Leaf 410: 2 ancestors
Leaf 514: 2 ancestors
Leaf 12: 2 ancestors
Leaf 69: 5 ancestors
Leaf 45: 5 ancestors
Leaf 109: 2 ancestors
Leaf 205: 2 a